# Notebook 7 — Stage 5: Inference Server (Optional Demo)
**MAI 656 — Natural Language Processing | Canadian University Dubai | Spring 2026**

This notebook launches a **vLLM OpenAI-compatible inference server** that serves your fine-tuned Arabic HTR model.

> ⚠️ **GPU required (≥24 GB VRAM).** Run this on the AWS g5.xlarge instance.  
> This notebook is **optional** — only needed if you want to demo the model as an API.

**Prerequisite:** LoRA adapter trained and saved in `models/lora_adapter/` (Notebook 5)

**Provides:** An OpenAI-compatible REST API at `http://localhost:8000`

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install vLLM

> Installation may take a few minutes. vLLM includes compiled CUDA kernels.

In [ ]:
!pip install vllm --quiet

## 3. Set Project Root

**Colab:** mounts Google Drive and reads from `MyDrive/nlp_project`.  
**Local:** update `LOCAL_PROJECT_ROOT` below to point to your project folder.

In [ ]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/nlp_project')
else:
    # ── Set this to your local project directory ──────────────────────────
    PROJECT_ROOT = Path(r'C:\Users\mitah\github_projects\nlp_project')
    # ──────────────────────────────────────────────────────────────────────

assert PROJECT_ROOT.exists(), (
    f'Project root not found: {PROJECT_ROOT}\n'
    'Update LOCAL_PROJECT_ROOT above to match your system.'
)

print(f'{"Colab" if IN_COLAB else "Local"} | Project root: {PROJECT_ROOT}')

## 4. Configure Server Parameters

> ⚠️ `--max-lora-rank 32` **must** match your adapter's rank (`LORA_R` from Notebook 5).  
> vLLM defaults to 16, which will crash with: `ValueError: LoRA rank 32 is greater than max_lora_rank 16`

In [ ]:
from pathlib import Path

BASE_MODEL   = 'Qwen/Qwen2.5-VL-7B-Instruct'
ADAPTER_DIR  = Path(PROJECT_ROOT) / 'models' / 'lora_adapter'
LORA_RANK    = 32     # must match training config
MAX_MODEL_LEN = 8192
PORT         = 8000
HOST         = '0.0.0.0'

SERVER_CMD = (
    f"python3 -m vllm.entrypoints.openai.api_server "
    f"--model {BASE_MODEL} "
    f"--enable-lora "
    f"--max-lora-rank {LORA_RANK} "
    f"--lora-modules arabic-ocr={ADAPTER_DIR} "
    f"--served-model-name arabic-ocr-v1 "
    f"--max-model-len {MAX_MODEL_LEN} "
    f"--gpu-memory-utilization 0.90 "
    f"--trust-remote-code "
    f"--host {HOST} --port {PORT} --dtype auto"
)

print('Server command:')
print(SERVER_CMD)

## 5. Launch the Server

The server runs as a **background process**. Output is logged to `logs/vllm_server.log`.

The server is ready when you see: `INFO:     Application startup complete.`

In [ ]:
import subprocess
import time
from pathlib import Path

LOG_DIR = Path(PROJECT_ROOT) / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
log_file = LOG_DIR / 'vllm_server.log'

print(f'Starting vLLM server... (logs → {log_file})')
server_proc = subprocess.Popen(
    SERVER_CMD,
    shell=True,
    stdout=open(log_file, 'w'),
    stderr=subprocess.STDOUT
)

print(f'Server PID: {server_proc.pid}')
print('Waiting for startup (this takes ~60–90 seconds)...')

# Poll until healthy
import urllib.request

for attempt in range(60):
    time.sleep(5)
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/health', timeout=3)
        print(f'\nServer is healthy after ~{(attempt+1)*5}s!')
        break
    except Exception:
        print(f'  attempt {attempt+1}/60...', end='\r')
else:
    print('\nServer did not start in time. Check logs/vllm_server.log')

## 6. Health Check

In [ ]:
!curl -s http://localhost:8000/health

## 7. List Available Models

In [ ]:
!curl -s http://localhost:8000/v1/models | python3 -m json.tool

## 8. Run a Test Inference

Replace `TEST_IMAGE_PATH` with the path to any image from your dataset.

In [ ]:
import base64
import json
import urllib.request
from pathlib import Path

TEST_IMAGE_PATH = Path(PROJECT_ROOT) / 'data' / 'raw'  # pick any image

# Find first available image
test_images = list(TEST_IMAGE_PATH.glob('*.png')) + list(TEST_IMAGE_PATH.glob('*.jpg'))
if not test_images:
    print('No test images found in data/raw/. Add a PNG or JPG to test.')
else:
    img_path = test_images[0]
    print(f'Testing with: {img_path.name}')

    with open(img_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode('utf-8')

    ext  = img_path.suffix.lower()
    mime = 'image/png' if ext == '.png' else 'image/jpeg'

    payload = json.dumps({
        'model': 'arabic-ocr-v1',
        'messages': [
            {'role': 'system',
             'content': 'Read the Arabic handwriting and return JSON.'},
            {'role': 'user', 'content': [
                {'type': 'image_url',
                 'image_url': {'url': f'data:{mime};base64,{b64}'}}
            ]}
        ],
        'max_tokens': 512
    }).encode()

    req = urllib.request.Request(
        f'http://localhost:{PORT}/v1/chat/completions',
        data=payload,
        headers={'Content-Type': 'application/json'},
    )

    with urllib.request.urlopen(req) as resp:
        result = json.loads(resp.read())

    print('\nRaw response:')
    print(result['choices'][0]['message']['content'])

## 9. View Server Logs

In [ ]:
!tail -50 logs/vllm_server.log

## 10. Stop the Server

In [ ]:
server_proc.terminate()
server_proc.wait()
print('vLLM server stopped.')

## Troubleshooting

| Problem | Cause | Fix |
|---------|-------|-----|
| `LoRA rank 32 > max_lora_rank 16` | vLLM default max rank is 16 | Pass `--max-lora-rank 32` |
| Server exits immediately | OOM or bad model path | Check `logs/vllm_server.log` |
| Port 8000 in use | Another process is using it | Change `PORT` or kill the other process |
| `Connection refused` | Server not ready yet | Wait longer; tail the log |
| Slow first inference | Model loading from disk | Subsequent calls are faster |

---

## Terminate Your AWS Instance!

> 🚨 **Don't forget! A running g5.xlarge costs ~$24/day.**

```bash
aws ec2 terminate-instances --instance-ids i-YOUR_INSTANCE_ID --region us-east-1
```